In [ ]:
from models import *
import torch
from dataloaders import *
import torch.nn as nn
from torch.utils.data import DataLoader
from shenghao_data import *

In [ ]:
class MarginToleranceLoss(nn.Module):
    def __init__(self, tol=1e-2, reduction='mean'):
        super().__init__()
        self.tol = tol
        self.reduction = reduction

    def forward(self, pred, y):
        loss = torch.relu(torch.abs(pred - y) - self.tol)
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss  # 'none'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = SimpleTransformerModel(d_model=64, n_heads=4, num_layers=4, tropical=True, num_classes=64, activation='relu',
                           tropical_attention_cls = TropicalAttention(64, 4, torch.device('cuda')), pool=True).to(device)

In [ ]:
ckpt_path = '15_exp/models/FloydWarshallDataset_tropical_0.0001_20000_20260417_041614_relu_best.pth'

state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)

In [ ]:
num_train_samples = 50000
num_val_samples = 10000
n = 8
low_train = 1
high_train = 15
low_test = 1
high_test = 15
use_integer = True
num_additional_node = 0
batch_size = 1
shuffle = True

In [ ]:
val_dataset = FloydWarshallDataset(num_val_samples, adversarial_range=(10, 20), length_range=(8, 8), noise_prob=0, value_range=(1, 15))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
criterion = MarginToleranceLoss()#nn.MSELoss()
model.eval()
val_loss = 0.0
n = 0
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        mask = torch.triu(torch.ones_like(y, dtype=torch.bool), diagonal=1)
        loss = criterion(out[mask], y[mask])
        val_loss += loss.item() * mask.sum().item()
        n += mask.sum().item()
val_loss /= n

In [ ]:
print(val_loss)